# Практическая работа 4
Выполнение задач обработки данных в распределённых вычислениях

Основная серия проводится при AQE=false, shuffle.partitions=8, autobroadcast=-1. Все времена измеряются внутри одной SparkSession в Ubuntu VirtualBox, а не берутся из методички. Самостоятельный вариант B: позиции заказов и продавцы, группировка по seller_state.

Студент: Албахтин И.В., ИНБО-12-23. Отдельная VM TIABD-Albakhtin-IV.

In [1]:
from pathlib import Path
import json
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark import StorageLevel
from lab_helpers import environment, records, save_json, plan, measured, checkpoint, ui_data, schema_signature
ROOT = Path.cwd()
STAGING = ROOT / "data" / "spark_input"
OUT = ROOT / "exports"
OUT.mkdir(exist_ok=True)
def read_csv(name, schema=None, infer=False):
    reader = spark.read.option("header", True).option("encoding", "UTF-8")
    if schema is not None: reader = reader.schema(schema)
    return reader.option("inferSchema", infer).csv(str(STAGING / f"{name}.csv"))

from time import perf_counter
from statistics import median
spark = (SparkSession.builder.master("local[*]").appName("TIABD_Practice_4_Albakhtin_IV")
    .config("spark.driver.memory", "1536m").config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false").config("spark.sql.autoBroadcastJoinThreshold", "-1")
    .config("spark.ui.bindAddress", "0.0.0.0").getOrCreate())
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext
result = {"environment":environment(spark),"source":json.loads((ROOT/"source_manifest.json").read_text()), "metrics":{}}
print(json.dumps(result["environment"],ensure_ascii=False,indent=2))


26/10/05 19:44:23 WARN Utils: Your hostname, tiabd-albakhtin-iv resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/05 19:44:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/05 19:44:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


{
  "python": "3.10.12",
  "java": "openjdk version \"17.0.20.1\" 2026-08-18",
  "spark": "3.5.7",
  "os": "Linux-5.15.0-198-generic-x86_64-with-glibc2.35",
  "cpu": "Architecture:                            x86_64\nCPU op-mode(s):                          32-bit, 64-bit\nAddress sizes:                           46 bits physical, 48 bits virtual\nByte Order:                              Little Endian\nCPU(s):                                  2\nOn-line CPU(s) list:                     0,1\nVendor ID:                               AuthenticAMD\nModel name:                              AMD Ryzen 7 9700X 8-Core Processor\nCPU family:                              26\nModel:                                   68\nThread(s) per core:                      1\nCore(s) per socket:                      2\nSocket(s):                               1\nStepping:                                0\nBogoMIPS:                                7599.98\nFlags:                                   fpu vme de pse t

## 1 Подготовка Parquet
Staging и перевод в Parquet исключены из замеров. Дополнительно выводятся реальные входные партиции.

In [2]:
PARQUET = ROOT/"data"/"spark_input_parquet"
names = ["geolocation","order_items","products","sellers","product_category_name_translation"]
for name in names:
    read_csv(name,infer=True).write.mode("overwrite").parquet(str(PARQUET/name))
geo, items, products, sellers, translations = [spark.read.parquet(str(PARQUET/name)) for name in names]
result["inputs"] = {name:{"rows":df.count(),"partitions":df.rdd.getNumPartitions()} for name,df in zip(names,[geo,items,products,sellers,translations])}
print(result["inputs"])
_, result["metrics"]["baseline"] = measured(spark,"P4_BASELINE",items.count)


{'geolocation': {'rows': 1000163, 'partitions': 2}, 'order_items': {'rows': 112650, 'partitions': 2}, 'products': {'rows': 32951, 'partitions': 1}, 'sellers': {'rows': 3095, 'partitions': 1}, 'product_category_name_translation': {'rows': 71, 'partitions': 1}}


P4_BASELINE {"seconds": 0.055194763000031344, "jobs": [25], "stages": [30, 31], "tasks": 3, "shuffle_read": 118, "shuffle_write": 118, "stage_metrics": [{"stageId": 31, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 118, "shuffleWriteBytes": 0}, {"stageId": 30, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 21482, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 118}]}


## 2 Lazy evaluation
Фиксируются Job ID перед созданием цепочки, после transformations и после count. Построение плана не должно запускать вычислительную Job.

In [3]:
sc.setJobGroup("P4_LAZY","Lazy evaluation")
before = sorted(sc.statusTracker().getJobIdsForGroup("P4_LAZY"))
prepared = (geo.select("geolocation_zip_code_prefix","geolocation_state","geolocation_lat","geolocation_lng")
    .filter(F.col("geolocation_state").isNotNull()).withColumn("abs_lat",F.abs("geolocation_lat")))
after_transform = sorted(sc.statusTracker().getJobIdsForGroup("P4_LAZY"))
n, metric = measured(spark,"P4_LAZY",prepared.count)
result["lazy"] = {"before":before,"after_transform":after_transform,"after_action":metric["jobs"],"rows":n,"seconds":metric["seconds"]}
assert before==after_transform
print(result["lazy"])


P4_LAZY {"seconds": 0.1422604260000071, "jobs": [26], "stages": [32, 33], "tasks": 3, "shuffle_read": 118, "shuffle_write": 118, "stage_metrics": [{"stageId": 33, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 118, "shuffleWriteBytes": 0}, {"stageId": 32, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 45110, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 118}]}
{'before': [], 'after_transform': [], 'after_action': [26], 'rows': 1000163, 'seconds': 0.1422604260000071}


## 3 Партиции и перекос
Диагностическая группировка сама является отдельным вычислением. Размеры партиций сравниваются на одинаковом наборе.

In [4]:
def profile(df):
    p = df.select(F.spark_partition_id().alias("pid")).groupBy("pid").count().orderBy("pid")
    values=records(p)
    sizes=[r["count"] for r in values]
    return {"partitions":df.rdd.getNumPartitions(),"sizes":values,"min":min(sizes),"max":max(sizes),"mean":sum(sizes)/len(sizes)}
state_counts=geo.groupBy("geolocation_state").count().orderBy(F.desc("count"))
state_counts.show(5,False)
result["state_top"]=records(state_counts.limit(5))
result["input_profile"]=profile(geo)
skewed16=geo.repartition(16,"geolocation_state")
repart4=skewed16.repartition(4)
coal4=skewed16.coalesce(4)
plan(repart4,"repartition4")
plan(coal4,"coalesce4")
result["profiles"]={"skewed16":profile(skewed16),"repartition4":profile(repart4),"coalesce4":profile(coal4)}
_,result["metrics"]["repartition4"]=measured(spark,"P4_REPARTITION",repart4.count)
_,result["metrics"]["coalesce4"]=measured(spark,"P4_COALESCE",coal4.count)
print(json.dumps(result["profiles"],indent=2))


+-----------------+------+
|geolocation_state|count |
+-----------------+------+
|SP               |404268|
|MG               |126336|
|RJ               |121169|
|RS               |61851 |
|PR               |57859 |
+-----------------+------+
only showing top 5 rows



== Physical Plan ==
Exchange (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [geolocation_zip_code_prefix#193, geolocation_lat#194, geolocation_lng#195, geolocation_city#196, geolocation_state#197]
Batched: true
Location: InMemoryFileIndex [file:/home/albakhtin/tiabd/АлбахтинИВ/Spark/data/spark_input_parquet/geolocation]
ReadSchema: struct<geolocation_zip_code_prefix:int,geolocation_lat:double,geolocation_lng:double,geolocation_city:string,geolocation_state:string>

(2) ColumnarToRow [codegen id : 1]
Input [5]: [geolocation_zip_code_prefix#193, geolocation_lat#194, geolocation_lng#195, geolocation_city#196, geolocation_state#197]

(3) Exchange
Input [5]: [geolocation_zip_code_prefix#193, geolocation_lat#194, geolocation_lng#195, geolocation_city#196, geolocation_state#197]
Arguments: RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=556]


== Physical Plan ==
Coalesce (4)
+- Exchange (3)
   +- * ColumnarToRow (2)
      +- Scan parquet  (1)


(1)

P4_REPARTITION {"seconds": 0.3282771149999917, "jobs": [37], "stages": [66, 67, 68, 69], "tasks": 23, "shuffle_read": 176717, "shuffle_write": 176717, "stage_metrics": [{"stageId": 69, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 236, "shuffleWriteBytes": 0}, {"stageId": 68, "numTasks": 4, "numCompleteTasks": 4, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 21232, "shuffleWriteBytes": 236}, {"stageId": 67, "numTasks": 16, "numCompleteTasks": 16, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 155249, "shuffleWriteBytes": 21232}, {"stageId": 66, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 34342, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 155249}]}


P4_COALESCE {"seconds": 0.192939583999987, "jobs": [38], "stages": [70, 71, 72], "tasks": 7, "shuffle_read": 155485, "shuffle_write": 155485, "stage_metrics": [{"stageId": 72, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 236, "shuffleWriteBytes": 0}, {"stageId": 71, "numTasks": 4, "numCompleteTasks": 4, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 155249, "shuffleWriteBytes": 236}, {"stageId": 70, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 34342, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 155249}]}
{
  "skewed16": {
    "partitions": 16,
    "sizes": [
      {
        "pid": 0,
        "count": 40594
      },
      {
        "pid": 3,
        "count": 154470
      },
      {
        "pid": 4,
        "count": 466119
      },
      {
        "pid": 5,
        "count": 121815
      },
      {
        "pid": 6,
        "count": 12863
      },
      {
        "pid": 7,
        "count": 25688
      },
    

## 4 GroupBy и physical plan
Частичная HashAggregate сокращает данные перед Exchange; финальная агрегация получает восемь shuffle partitions.

In [5]:
seller_stats=items.groupBy("seller_id").agg(F.count("*").alias("items"),F.round(F.sum("price"),2).alias("revenue"),F.round(F.avg("price"),2).alias("avg_price")).orderBy(F.desc("revenue"))
top, result["metrics"]["groupby"] = measured(spark,"P4_GROUPBY",lambda:records(seller_stats.limit(5)))
result["seller_top"]=top
seller_stats.show(5,False)
plan(seller_stats,"groupby")
demo=(items.filter("price>=100").select("seller_id","price","freight_value").withColumn("gross",F.col("price")+F.col("freight_value"))
    .groupBy("seller_id").agg(F.round(F.sum("gross"),2).alias("gross_sum")))
plan(demo,"operator_demo")


P4_GROUPBY {"seconds": 0.20747943400010627, "jobs": [39], "stages": [73, 74], "tasks": 10, "shuffle_read": 269271, "shuffle_write": 269271, "stage_metrics": [{"stageId": 74, "numTasks": 8, "numCompleteTasks": 8, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 269271, "shuffleWriteBytes": 0}, {"stageId": 73, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 819626, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 269271}]}
+--------------------------------+-----+---------+---------+
|seller_id                       |items|revenue  |avg_price|
+--------------------------------+-----+---------+---------+
|4869f7a5dfa277a7dca6462dcf3b52b2|1156 |229472.63|198.51   |
|53243585a1d6dc2643021fd1853d8905|410  |222776.05|543.36   |
|4a3ca9315b744ce9f8e9374361493884|1987 |200472.92|100.89   |
|fa1c13f2614d7b5c4749cbc52fecda94|586  |194042.03|331.13   |
|7c67e1448b00f6e969d365cea6b010ab|1364 |187923.89|137.77   |
+--------------------------------+-----+---------+---------+
onl

'== Physical Plan ==\n* HashAggregate (7)\n+- Exchange (6)\n   +- * HashAggregate (5)\n      +- * Project (4)\n         +- * Filter (3)\n            +- * ColumnarToRow (2)\n               +- Scan parquet  (1)\n\n\n(1) Scan parquet \nOutput [3]: [seller_id#206, price#208, freight_value#209]\nBatched: true\nLocation: InMemoryFileIndex [file:/home/albakhtin/tiabd/АлбахтинИВ/Spark/data/spark_input_parquet/order_items]\nPushedFilters: [IsNotNull(price), GreaterThanOrEqual(price,100.0)]\nReadSchema: struct<seller_id:string,price:double,freight_value:double>\n\n(2) ColumnarToRow [codegen id : 1]\nInput [3]: [seller_id#206, price#208, freight_value#209]\n\n(3) Filter [codegen id : 1]\nInput [3]: [seller_id#206, price#208, freight_value#209]\nCondition : (isnotnull(price#208) AND (price#208 >= 100.0))\n\n(4) Project [codegen id : 1]\nOutput [2]: [seller_id#206, (price#208 + freight_value#209) AS gross#469]\nInput [3]: [seller_id#206, price#208, freight_value#209]\n\n(5) HashAggregate [codegen i

## 5 Sort Merge Join и Broadcast Hash Join
JOIN по product_id одинаков по смыслу. Сравниваются число строк, планы и фактические счётчики Spark UI. Дополнительно проводятся три прогретых повторения обеих стратегий.

In [6]:
smj=items.hint("merge").join(products.hint("merge"),"product_id","inner")
bhj=items.join(F.broadcast(products),"product_id","inner")
plan(smj,"smj")
plan(bhj,"bhj")
smj_n,result["metrics"]["smj"]=measured(spark,"P4_SMJ",smj.count)
bhj_n,result["metrics"]["bhj"]=measured(spark,"P4_BHJ",bhj.count)
assert smj_n==bhj_n==items.count()
join_runs={"smj":[],"bhj":[]}
for _ in range(3):
    for name,df in [("smj",smj),("bhj",bhj)]:
        t=perf_counter(); df.count(); join_runs[name].append(perf_counter()-t)
result["joins"]={"rows":smj_n,"runs":join_runs,"medians":{k:median(v) for k,v in join_runs.items()}}
print(result["joins"])
checkpoint("pr4_joins",spark)


== Physical Plan ==
* Project (12)
+- * SortMergeJoin Inner (11)
   :- * Sort (5)
   :  +- Exchange (4)
   :     +- * Filter (3)
   :        +- * ColumnarToRow (2)
   :           +- Scan parquet  (1)
   +- * Sort (10)
      +- Exchange (9)
         +- * Filter (8)
            +- * ColumnarToRow (7)
               +- Scan parquet  (6)


(1) Scan parquet 
Output [7]: [order_id#203, order_item_id#204, product_id#205, seller_id#206, shipping_limit_date#207, price#208, freight_value#209]
Batched: true
Location: InMemoryFileIndex [file:/home/albakhtin/tiabd/АлбахтинИВ/Spark/data/spark_input_parquet/order_items]
PushedFilters: [IsNotNull(product_id)]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:double,freight_value:double>

(2) ColumnarToRow [codegen id : 1]
Input [7]: [order_id#203, order_item_id#204, product_id#205, seller_id#206, shipping_limit_date#207, price#208, freight_value#209]

(3) Filter [codegen id : 1]

P4_SMJ {"seconds": 0.33832371500000136, "jobs": [41], "stages": [77, 78, 79, 80], "tasks": 12, "shuffle_read": 3917308, "shuffle_write": 3917308, "stage_metrics": [{"stageId": 80, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 472, "shuffleWriteBytes": 0}, {"stageId": 79, "numTasks": 8, "numCompleteTasks": 8, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 3916836, "shuffleWriteBytes": 472}, {"stageId": 78, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 1141848, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 1114450}, {"stageId": 77, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 1687040, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 2802386}]}


P4_BHJ {"seconds": 0.284069240000008, "jobs": [42, 43], "stages": [81, 82, 83], "tasks": 4, "shuffle_read": 118, "shuffle_write": 118, "stage_metrics": [{"stageId": 83, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 118, "shuffleWriteBytes": 0}, {"stageId": 82, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 1687040, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 118}, {"stageId": 81, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 1141848, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 0}]}


{'rows': 112650, 'runs': {'smj': [0.21792677600001298, 0.2296854109999913, 0.21326704400007657], 'bhj': [0.12499758600006317, 0.14419666699996014, 0.1349928980000641]}, 'medians': {'smj': 0.21792677600001298, 'bhj': 0.1349928980000641}}
SCREENSHOT CHECKPOINT pr4_joins


## 6 Persist и повторные действия
Сравнение включает цену материализации. Действия вычисляют контрольные суммы, чтобы оптимизатор не выбросил вычисляемое поле.

In [7]:
spark.catalog.clearCache()
joined=(items.join(F.broadcast(products.select("product_id","product_category_name")),"product_id","left")
    .join(F.broadcast(sellers.select("seller_id","seller_state")),"seller_id","left")
    .withColumn("gross_value",F.col("price")+F.col("freight_value")))
def checksum(df,key):
    return df.groupBy(key).agg(F.sum("gross_value").alias("gross")).agg(F.sum("gross").alias("checksum")).first()[0]
a,m1=measured(spark,"P4_NOCACHE",lambda:checksum(joined,"product_category_name"))
b,m2=measured(spark,"P4_NOCACHE",lambda:checksum(joined,"seller_state"))
cached=joined.persist(StorageLevel.MEMORY_AND_DISK)
cached_n,mat=measured(spark,"P4_PERSIST",cached.count)
c,m3=measured(spark,"P4_PERSIST",lambda:checksum(cached,"product_category_name"))
d,m4=measured(spark,"P4_PERSIST",lambda:checksum(cached,"seller_state"))
assert abs(a-c)<0.01 and abs(b-d)<0.01
result["cache"]={"rows":cached_n,"no_cache":[m1,m2],"materialization":mat,"cached":[m3,m4],"checksums":[a,b,c,d],"storage":ui_data(spark,"/storage/rdd")}
print(json.dumps(result["cache"],indent=2))
result["metrics"]["persist_materialization"]=mat
result["metrics"]["persist_repeat"]=m3
checkpoint("pr4_storage",spark,["/storage/","/SQL/","/stages/"])
cached.unpersist(blocking=True)
print("Cached after unpersist:",cached.is_cached)


P4_NOCACHE {"seconds": 0.3434797709999202, "jobs": [54, 55, 56], "stages": [107, 108, 109, 110, 111], "tasks": 13, "shuffle_read": 6445, "shuffle_write": 6445, "stage_metrics": [{"stageId": 111, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 472, "shuffleWriteBytes": 0}, {"stageId": 110, "numTasks": 8, "numCompleteTasks": 8, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 5973, "shuffleWriteBytes": 472}, {"stageId": 109, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 2461575, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 5973}, {"stageId": 108, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 1141848, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 0}, {"stageId": 107, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 131816, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 0}]}


P4_NOCACHE {"seconds": 0.3213934949999384, "jobs": [57, 58, 59], "stages": [112, 113, 114, 115, 116], "tasks": 13, "shuffle_read": 2088, "shuffle_write": 2088, "stage_metrics": [{"stageId": 116, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 470, "shuffleWriteBytes": 0}, {"stageId": 115, "numTasks": 8, "numCompleteTasks": 8, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 1618, "shuffleWriteBytes": 470}, {"stageId": 114, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 2461575, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 1618}, {"stageId": 113, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 131816, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 0}, {"stageId": 112, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 1141848, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 0}]}


P4_PERSIST {"seconds": 0.663197351000008, "jobs": [60, 61, 62], "stages": [117, 118, 119, 120], "tasks": 5, "shuffle_read": 118, "shuffle_write": 118, "stage_metrics": [{"stageId": 120, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 118, "shuffleWriteBytes": 0}, {"stageId": 119, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 6677415, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 118}, {"stageId": 118, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 131816, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 0}, {"stageId": 117, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 1141848, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 0}]}


P4_PERSIST {"seconds": 0.21483621600009428, "jobs": [63], "stages": [121, 122, 123], "tasks": 11, "shuffle_read": 6445, "shuffle_write": 6445, "stage_metrics": [{"stageId": 123, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 472, "shuffleWriteBytes": 0}, {"stageId": 122, "numTasks": 8, "numCompleteTasks": 8, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 5973, "shuffleWriteBytes": 472}, {"stageId": 121, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 8727795, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 5973}]}
P4_PERSIST {"seconds": 0.1163638540001557, "jobs": [64], "stages": [124, 125, 126], "tasks": 11, "shuffle_read": 2088, "shuffle_write": 2088, "stage_metrics": [{"stageId": 126, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 470, "shuffleWriteBytes": 0}, {"stageId": 125, "numTasks": 8, "numCompleteTasks": 8, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes"

Cached after unpersist: False


## 7 Benchmark
N=2 000 000, key=id % 1000, value=sqrt(id+1). После одного прогрева для каждого p выполняются три запуска; число shuffle partitions неизменно и равно восьми.

In [8]:
N=2_000_000
def run_benchmark(p,repeats=3):
    times=[]; checks=[]
    for _ in range(repeats):
        df=spark.range(0,N,1,numPartitions=p).withColumn("key",(F.col("id")%1000).cast("int")).withColumn("value",F.sqrt(F.col("id")+1.0))
        t=perf_counter()
        check=df.groupBy("key").agg(F.sum("value").alias("s")).agg(F.sum("s")).first()[0]
        times.append(perf_counter()-t); checks.append(check)
    return {"p":p,"times":times,"median":median(times),"checksum":checks}
sc.setJobGroup("P4_BENCHMARK","Synthetic partition benchmark")
run_benchmark(4,1)
result["benchmark"]=[run_benchmark(p) for p in [1,2,4,8,16,32]]
for row in result["benchmark"]: row["speedup"]=result["benchmark"][0]["median"]/row["median"]
print(json.dumps(result["benchmark"],indent=2))
geo8=geo.repartition(8)
q_results=[]
for q in [2,4,8,16,32]:
    spark.conf.set("spark.sql.shuffle.partitions",str(q))
    t=perf_counter(); rows=geo8.groupBy("geolocation_state").count().count(); seconds=perf_counter()-t
    q_results.append({"q":q,"groups":rows,"seconds":seconds})
spark.conf.set("spark.sql.shuffle.partitions","8")
result["shuffle_q"]=q_results
print(q_results)


[
  {
    "p": 1,
    "times": [
      0.13344388300015453,
      0.05495223499997337,
      0.059042877000138105
    ],
    "median": 0.059042877000138105,
    "checksum": [
      1885618790.063052,
      1885618790.063052,
      1885618790.063052
    ],
    "speedup": 1.0
  },
  {
    "p": 2,
    "times": [
      0.19455228199990415,
      0.08966217800002596,
      0.10353810600008728
    ],
    "median": 0.10353810600008728,
    "checksum": [
      1885618790.0630507,
      1885618790.0630507,
      1885618790.0630507
    ],
    "speedup": 0.5702526275696828
  },
  {
    "p": 4,
    "times": [
      0.13158814600001278,
      0.09018109499993443,
      0.10163243700003477
    ],
    "median": 0.10163243700003477,
    "checksum": [
      1885618790.0630512,
      1885618790.0630512,
      1885618790.0630512
    ],
    "speedup": 0.5809452055166
  },
  {
    "p": 8,
    "times": [
      0.21208750699997836,
      0.11176324100006241,
      0.10913225300009799
    ],
    "median": 0.1

[{'q': 2, 'groups': 27, 'seconds': 0.34120527000004586}, {'q': 4, 'groups': 27, 'seconds': 0.21323536500017326}, {'q': 8, 'groups': 27, 'seconds': 0.282081479999988}, {'q': 16, 'groups': 27, 'seconds': 0.278198853000049}, {'q': 32, 'groups': 27, 'seconds': 0.28126977399983843}]


## 8 Маленькие файлы
Число файлов и их размер измеряются в файловой системе после фактической записи, а не предсказываются по числу партиций.

In [9]:
SMALL=ROOT/"small_files_experiment"
def write_measure(df,p,tag):
    path=SMALL/tag
    _,metrics=measured(spark,"P4_FILES_"+tag,lambda:df.repartition(p).write.mode("overwrite").parquet(str(path)))
    files=list(path.glob("part-*.parquet"))
    sizes=[f.stat().st_size for f in files]
    return {"p":p,"files":len(files),"total_mib":sum(sizes)/1024**2,"mean_kib":sum(sizes)/len(sizes)/1024,"metrics":metrics}
result["small_files"]=[write_measure(items,p,f"items_{p}") for p in [1,4,16,64]]
items.coalesce(1).write.mode("overwrite").parquet(str(SMALL/"coalesce1"))
print(json.dumps(result["small_files"],indent=2))
spark.conf.set("spark.sql.adaptive.enabled","true")
spark.conf.set("spark.sql.shuffle.partitions","32")
adaptive=geo.groupBy("geolocation_state").count()
adaptive.collect()
result["aqe"]={"groups":len(adaptive.collect()),"output_partitions":adaptive.rdd.getNumPartitions()}
plan(adaptive,"aqe_final")
spark.conf.set("spark.sql.adaptive.enabled","false")
spark.conf.set("spark.sql.shuffle.partitions","8")
print(result["aqe"])


P4_FILES_items_1 {"seconds": 0.3131553250000252, "jobs": [89], "stages": [198, 199], "tasks": 3, "shuffle_read": 10751186, "shuffle_write": 10751186, "stage_metrics": [{"stageId": 199, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 6770099, "shuffleReadBytes": 10751186, "shuffleWriteBytes": 0}, {"stageId": 198, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 6677415, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 10751186}]}


P4_FILES_items_4 {"seconds": 0.3151144229998408, "jobs": [90], "stages": [200, 201], "tasks": 6, "shuffle_read": 12012709, "shuffle_write": 12012709, "stage_metrics": [{"stageId": 201, "numTasks": 4, "numCompleteTasks": 4, "inputBytes": 0, "outputBytes": 7827036, "shuffleReadBytes": 12012709, "shuffleWriteBytes": 0}, {"stageId": 200, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 6677415, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 12012709}]}


P4_FILES_items_16 {"seconds": 0.45948915700000725, "jobs": [91], "stages": [202, 203], "tasks": 18, "shuffle_read": 12022984, "shuffle_write": 12022984, "stage_metrics": [{"stageId": 203, "numTasks": 16, "numCompleteTasks": 16, "inputBytes": 0, "outputBytes": 9181711, "shuffleReadBytes": 12022984, "shuffleWriteBytes": 0}, {"stageId": 202, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 6677415, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 12022984}]}


P4_FILES_items_64 {"seconds": 1.0146044079999683, "jobs": [92], "stages": [204, 205], "tasks": 66, "shuffle_read": 12065531, "shuffle_write": 12065531, "stage_metrics": [{"stageId": 205, "numTasks": 64, "numCompleteTasks": 64, "inputBytes": 0, "outputBytes": 10801884, "shuffleReadBytes": 12065531, "shuffleWriteBytes": 0}, {"stageId": 204, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 6677415, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 12065531}]}


[
  {
    "p": 1,
    "files": 1,
    "total_mib": 6.456469535827637,
    "mean_kib": 6611.4248046875,
    "metrics": {
      "seconds": 0.3131553250000252,
      "jobs": [
        89
      ],
      "stages": [
        198,
        199
      ],
      "tasks": 3,
      "shuffle_read": 10751186,
      "shuffle_write": 10751186,
      "stage_metrics": [
        {
          "stageId": 199,
          "numTasks": 1,
          "numCompleteTasks": 1,
          "inputBytes": 0,
          "outputBytes": 6770099,
          "shuffleReadBytes": 10751186,
          "shuffleWriteBytes": 0
        },
        {
          "stageId": 198,
          "numTasks": 2,
          "numCompleteTasks": 2,
          "inputBytes": 6677415,
          "outputBytes": 0,
          "shuffleReadBytes": 0,
          "shuffleWriteBytes": 10751186
        }
      ]
    }
  },
  {
    "p": 4,
    "files": 4,
    "total_mib": 7.464443206787109,
    "mean_kib": 1910.8974609375,
    "metrics": {
      "seconds": 0.31511442299984

## 9 Самостоятельный вариант B
order_items ↔ sellers, категориальный ключ seller_state. Повторяются lazy, repartition/coalesce, groupBy, две стратегии JOIN, persist, медианы и запись.

In [10]:
sc.setJobGroup("P4_B_LAZY","Independent B lazy")
before=sorted(sc.statusTracker().getJobIdsForGroup("P4_B_LAZY"))
b_prepared=items.select("seller_id","price","freight_value").filter("price>0").withColumn("gross",F.col("price")+F.col("freight_value"))
after=sorted(sc.statusTracker().getJobIdsForGroup("P4_B_LAZY"))
b_rows,b_lazy=measured(spark,"P4_B_LAZY",b_prepared.count)
assert before==after
b_smj=b_prepared.hint("merge").join(sellers.hint("merge"),"seller_id","inner")
b_bhj=b_prepared.join(F.broadcast(sellers),"seller_id","inner")
plan(b_smj,"b_smj"); plan(b_bhj,"b_bhj")
b_sn,b_sm=measured(spark,"P4_B_SMJ",b_smj.count)
b_bn,b_bm=measured(spark,"P4_B_BHJ",b_bhj.count)
assert b_sn==b_bn==b_rows
b_group=b_bhj.groupBy("seller_state").agg(F.count("*").alias("lines"),F.round(F.sum("gross"),2).alias("gross"))
b_values,b_gm=measured(spark,"P4_B_GROUPBY",lambda:records(b_group.orderBy(F.desc("gross"))))
plan(b_group,"b_groupby")
b_skew=b_bhj.repartition(16,"seller_state")
b_repart=b_skew.repartition(4); b_coal=b_skew.coalesce(4)
plan(b_repart,"b_repartition"); plan(b_coal,"b_coalesce")
b_profiles={"repartition":profile(b_repart),"coalesce":profile(b_coal)}
def b_action(df,key):
    return df.groupBy(key).agg(F.sum("gross").alias("s")).agg(F.sum("s")).first()[0]
t=perf_counter(); ba=b_action(b_bhj,"seller_state"); t1=perf_counter(); bb=b_action(b_bhj,"seller_city"); t2=perf_counter()
b_cache=b_bhj.persist(StorageLevel.MEMORY_AND_DISK)
t3=perf_counter(); b_cache.count(); t4=perf_counter(); bc=b_action(b_cache,"seller_state"); t5=perf_counter(); bd=b_action(b_cache,"seller_city"); t6=perf_counter()
assert max(abs(ba-bc),abs(bb-bd))<0.01
b_cache_times={"without":[t1-t,t2-t1],"materialization":t4-t3,"cached":[t5-t4,t6-t5]}
checkpoint("pr4_variant_storage",spark,["/storage/","/SQL/","/stages/"])
b_cache.unpersist(blocking=True)
b_benchmark=[]
for p in [1,2,4,8]:
    times=[]
    for repeat in range(4):
        variant=b_bhj.repartition(p).groupBy("seller_state").agg(F.sum("gross").alias("g"))
        t=perf_counter(); values=variant.collect(); dt=perf_counter()-t
        if repeat: times.append(dt)
    b_benchmark.append({"p":p,"times":times,"median":median(times)})
b_files=[write_measure(b_group,p,f"variant_{p}") for p in [1,4]]
b_group.write.mode("overwrite").parquet(str(OUT/"pr4_variant_parquet"))
result["variant"]={"name":"B","inputs":{"items":result["inputs"]["order_items"],"sellers":result["inputs"]["sellers"]},"lazy":{"before":before,"after_transform":after,"after_action":b_lazy["jobs"],"rows":b_rows},"join_rows":b_sn,"smj":b_sm,"bhj":b_bm,"groupby":b_gm,"groups":b_values,"profiles":b_profiles,"cache":b_cache_times,"benchmark":b_benchmark,"files":b_files}
print(json.dumps(result["variant"],ensure_ascii=False,indent=2))


P4_B_LAZY {"seconds": 0.04532825999990564, "jobs": [97], "stages": [212, 213], "tasks": 3, "shuffle_read": 118, "shuffle_write": 118, "stage_metrics": [{"stageId": 213, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 118, "shuffleWriteBytes": 0}, {"stageId": 212, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 491124, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 118}]}
== Physical Plan ==
* Project (13)
+- * SortMergeJoin Inner (12)
   :- * Sort (6)
   :  +- Exchange (5)
   :     +- * Project (4)
   :        +- * Filter (3)
   :           +- * ColumnarToRow (2)
   :              +- Scan parquet  (1)
   +- * Sort (11)
      +- Exchange (10)
         +- * Filter (9)
            +- * ColumnarToRow (8)
               +- Scan parquet  (7)


(1) Scan parquet 
Output [3]: [seller_id#206, price#208, freight_value#209]
Batched: true
Location: InMemoryFileIndex [file:/home/albakhtin/tiabd/АлбахтинИВ/Spark/data/spark_input_parquet/

P4_B_SMJ {"seconds": 0.19172874699984277, "jobs": [98], "stages": [214, 215, 216, 217], "tasks": 12, "shuffle_read": 1369691, "shuffle_write": 1369691, "stage_metrics": [{"stageId": 217, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 472, "shuffleWriteBytes": 0}, {"stageId": 216, "numTasks": 8, "numCompleteTasks": 8, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 1369219, "shuffleWriteBytes": 472}, {"stageId": 215, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 964388, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 1262840}, {"stageId": 214, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 131816, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 106379}]}
P4_B_BHJ {"seconds": 0.08772391100001187, "jobs": [99, 100], "stages": [218, 219, 220], "tasks": 4, "shuffle_read": 118, "shuffle_write": 118, "stage_metrics": [{"stageId": 220, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes"

P4_B_GROUPBY {"seconds": 0.2077453179999793, "jobs": [101, 102, 103], "stages": [221, 222, 223, 224, 225, 226], "tasks": 27, "shuffle_read": 5322, "shuffle_write": 3472, "stage_metrics": [{"stageId": 226, "numTasks": 8, "numCompleteTasks": 8, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 1622, "shuffleWriteBytes": 0}, {"stageId": 225, "numTasks": 8, "numCompleteTasks": 8, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 1850, "shuffleWriteBytes": 1622}, {"stageId": 223, "numTasks": 8, "numCompleteTasks": 8, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 1850, "shuffleWriteBytes": 0}, {"stageId": 222, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 1086721, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 1850}, {"stageId": 221, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 131816, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 0}]}
== Physical Plan ==
* HashAggregate (13)
+- Exchange (12)
   +- * HashAggregate (11)
      +- *

SCREENSHOT CHECKPOINT pr4_variant_storage


P4_FILES_variant_1 {"seconds": 0.1133840599998166, "jobs": [152, 153], "stages": [324, 325, 326, 327], "tasks": 12, "shuffle_read": 2800, "shuffle_write": 2800, "stage_metrics": [{"stageId": 327, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 0, "outputBytes": 1367, "shuffleReadBytes": 950, "shuffleWriteBytes": 0}, {"stageId": 326, "numTasks": 8, "numCompleteTasks": 8, "inputBytes": 0, "outputBytes": 0, "shuffleReadBytes": 1850, "shuffleWriteBytes": 950}, {"stageId": 325, "numTasks": 2, "numCompleteTasks": 2, "inputBytes": 1086721, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 1850}, {"stageId": 324, "numTasks": 1, "numCompleteTasks": 1, "inputBytes": 131816, "outputBytes": 0, "shuffleReadBytes": 0, "shuffleWriteBytes": 0}]}
P4_FILES_variant_4 {"seconds": 0.13393151200011744, "jobs": [154, 155], "stages": [328, 329, 330, 331], "tasks": 15, "shuffle_read": 3534, "shuffle_write": 3534, "stage_metrics": [{"stageId": 331, "numTasks": 4, "numCompleteTasks": 4, "inputByt

{
  "name": "B",
  "inputs": {
    "items": {
      "rows": 112650,
      "partitions": 2
    },
    "sellers": {
      "rows": 3095,
      "partitions": 1
    }
  },
  "lazy": {
    "before": [],
    "after_transform": [],
    "after_action": [
      97
    ],
    "rows": 112650
  },
  "join_rows": 112650,
  "smj": {
    "seconds": 0.19172874699984277,
    "jobs": [
      98
    ],
    "stages": [
      214,
      215,
      216,
      217
    ],
    "tasks": 12,
    "shuffle_read": 1369691,
    "shuffle_write": 1369691,
    "stage_metrics": [
      {
        "stageId": 217,
        "numTasks": 1,
        "numCompleteTasks": 1,
        "inputBytes": 0,
        "outputBytes": 0,
        "shuffleReadBytes": 472,
        "shuffleWriteBytes": 0
      },
      {
        "stageId": 216,
        "numTasks": 8,
        "numCompleteTasks": 8,
        "inputBytes": 0,
        "outputBytes": 0,
        "shuffleReadBytes": 1369219,
        "shuffleWriteBytes": 472
      },
      {
        "stageI

## 10 Журнал Spark UI и завершение
REST API текущего приложения сохраняет Jobs и Stages. Скриншоты снимаются до spark.stop. Измерения относятся только к этому стенду и текущим данным.

In [11]:
save_json("pr4.json",result)
save_json("pr4_jobs.json",ui_data(spark,"/jobs"))
save_json("pr4_stages.json",ui_data(spark,"/stages"))
save_json("pr4_sql.json",ui_data(spark,"/sql"))
checkpoint("pr4_completed",spark,["/jobs/","/stages/","/SQL/","/environment/"])


SCREENSHOT CHECKPOINT pr4_completed


## Выводы и ограничения
При интерпретации сопоставляются медианы, планы и UI. Exchange указывает на перераспределение; Coalesce сохраняет перекос. Broadcast меняет механизм JOIN без изменения строк. Цена persist включает материализацию. Рост p создаёт дополнительные задачи и файлы. AQE способен сократить число выходных shuffle partitions.

Ограничения: два vCPU одной VM не моделируют сеть физического кластера; JVM/JIT, файловый кэш и фоновые службы влияют на короткие замеры. Причинный эффект способа JOIN нельзя выводить только из одного времени запуска.

In [12]:
spark.stop()
print("SparkSession stopped")

SparkSession stopped
